In [18]:
import pandas as pd

In [19]:
dataset = pd.read_csv('bank_exit_or_not.csv')
dataset

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [20]:
## Remove the columns that are not required
dataset = dataset.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
dataset

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [21]:
# Categorize the data
from sklearn.preprocessing import OneHotEncoder

# Let's convert Geography by OneHotEncoding
onehotencoder = OneHotEncoder()
geography_fit_encoded = onehotencoder.fit_transform(dataset[['Geography']])
geography_fit_encoded_array = geography_fit_encoded.toarray()
geography_fit_encoded_df = pd.DataFrame(geography_fit_encoded_array, columns=onehotencoder.get_feature_names_out(['Geography']))

dataset = pd.concat([dataset, geography_fit_encoded_df], axis=1).drop(['Geography'], axis=1)


In [22]:

# Let's convert boolean to 0 and 1 (Gender)
from sklearn.preprocessing import LabelEncoder
labelencoder = LabelEncoder()
gender_fit_encoded = labelencoder.fit_transform(dataset['Gender'])
gender_fit_encoded_df = pd.DataFrame(gender_fit_encoded, columns=['Gender_Encoded'])
dataset = pd.concat([dataset, gender_fit_encoded_df], axis=1).drop(['Gender'], axis=1)

dataset

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Encoded
0,619,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,0
1,608,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,0
2,502,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,0
3,699,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,0
4,850,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,771,39,5,0.00,2,1,0,96270.64,0,1.0,0.0,0.0,1
9996,516,35,10,57369.61,1,1,1,101699.77,0,1.0,0.0,0.0,1
9997,709,36,7,0.00,1,0,1,42085.58,1,1.0,0.0,0.0,0
9998,772,42,3,75075.31,2,1,0,92888.52,1,0.0,1.0,0.0,1


In [23]:
# Save the encoders for future use
import pickle
with open('gender_fit_encoder.pkl', 'wb') as file:
    pickle.dump(labelencoder, file)
with open('geography_fit_encoder.pkl', 'wb') as file:
    pickle.dump(onehotencoder, file)

In [24]:
# Split the data into training and testing
from sklearn.model_selection import train_test_split
X = dataset.drop('Exited', axis=1)
y = dataset['Exited']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# Normalize the data
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [25]:
# Save the scaler for future use
with open('scaler.pkl', 'wb') as file:
    pickle.dump(sc, file)

In [26]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras import Input
import datetime

In [27]:
X_train

array([[ 0.16958176, -0.46460796,  0.00666099, ..., -0.5698444 ,
         1.74309049, -1.09168714],
       [-2.30455945,  0.30102557, -1.37744033, ...,  1.75486502,
        -0.57369368,  0.91601335],
       [-1.19119591, -0.94312892, -1.031415  , ..., -0.5698444 ,
        -0.57369368, -1.09168714],
       ...,
       [ 0.9015152 , -0.36890377,  0.00666099, ..., -0.5698444 ,
        -0.57369368,  0.91601335],
       [-0.62420521, -0.08179119,  1.39076231, ..., -0.5698444 ,
         1.74309049, -1.09168714],
       [-0.28401079,  0.87525072, -1.37744033, ...,  1.75486502,
        -0.57369368, -1.09168714]])

In [28]:
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation="sigmoid")
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
model.compile(optimizer='adam', loss="binary_crossentropy", metrics=['accuracy'])

In [30]:
logs_dir = "logs/fit"
tensorflow_callback  = TensorBoard(log_dir=logs_dir, histogram_freq=1)

In [31]:
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [32]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test), 
    epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8000 - loss: 0.4797 - val_accuracy: 0.8320 - val_loss: 0.4048
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8299 - loss: 0.3970 - val_accuracy: 0.8595 - val_loss: 0.3572
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8519 - loss: 0.3613 - val_accuracy: 0.8610 - val_loss: 0.3502
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8580 - loss: 0.3447 - val_accuracy: 0.8570 - val_loss: 0.3469
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8573 - loss: 0.3533 - val_accuracy: 0.8615 - val_loss: 0.3427
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8587 - loss: 0.3355 - val_accuracy: 0.8570 - val_loss: 0.3449
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8645 - loss: 0.3320 - val_accuracy: 0.8605 - val_loss: 0.3435
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8645 - loss: 0.3303 - val_accu

In [33]:
model.save('model.keras')